# One view from one GeoJSON

The smallest useful thing this package does: hand it **a polygon and one month**, get back
**one image of that month**. Everything else in this repo — ten-year animations, thermal
sharpening, stem-density overlays — is this, repeated and dressed up.

The notebook does it twice over, because the two answers are not the same thing:

1. **A still to look at** — colorised, annotated, legend, AOI outline. One `animate()` call.
2. **The same month as data** — a georeferenced GeoTIFF of the composited values, no furniture.

Runtime: a couple of minutes, almost all of it Earth Engine.

## 1 · Setup

Works from a clone (`pip install -e .`) or in Colab, which installs the package from the branch.

In [ ]:
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
URL = "gee_animation @ git+https://github.com/cwinkelmann/GEE_Animation.git@main"
if IS_COLAB:
    !pip install -q "{URL}"
    !pip install -q --force-reinstall --no-deps "{URL}"   # refresh our code on re-runs

# Repo root, so the bundled AOI files resolve wherever the notebook is started from.
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "aoi").is_dir()), None)
print("colab:", IS_COLAB, "| repo:", REPO)

## 2 · Authenticate (once)

`auth.init` reuses cached credentials, or a service-account key via `EE_SERVICE_ACCOUNT_KEY`. It prompts `ee.Authenticate()` only if neither is there.

In [ ]:
from gee_animation import auth

PROJECT = "hnee-331218"   # <-- your own Earth Engine project id
auth.init(PROJECT)

## 3 · The area of interest

A **path to a GeoJSON file is all `animate` needs**. It normalises the region for you
(`api._to_region_aoi`), which also accepts a raw GeoJSON dict, a `.shp`/`.zip`, or a plain
`[west, south, east, north]` bbox — so nothing below is specific to GeoJSON.

Two geometries matter and they are not the same:

| | what it is | what it does |
|---|---|---|
| **region** | your polygon, as given | gets outlined on the frame (`draw_region`), and is the AOI for cloud filtering and statistics |
| **frame** | the region's bounding box grown by `buffer_m` | the extent actually rendered |

So the picture is always **wider than your polygon**. That is deliberate: a windthrow patch
means nothing without the untouched forest around it.

In [ ]:
from gee_animation import aoi

if REPO is not None:
    REGION = str(REPO / "docs" / "aoi" / "r12" / "r12_footprint.geojson")
else:   # Colab: fetch the same file
    import urllib.request
    RAW = ("https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/main"
           "/docs/aoi/r12/r12_footprint.geojson")
    urllib.request.urlretrieve(RAW, "r12_footprint.geojson")
    REGION = "r12_footprint.geojson"

# --- your own AOI instead of the line above ---
#   REGION = "/path/to/your.geojson"        # any GeoJSON file (or a .shp / .zip)
#   REGION = [13.80, 52.90, 14.00, 53.05]   # or a bbox, no file needed: [W, S, E, N]

BUFFER_M = 1000.0

region_geom = aoi.parse({"geojson": REGION}) if isinstance(REGION, str) else aoi.parse({"bbox": REGION})
frame_bbox = aoi.frame_bbox_from_region(region_geom, BUFFER_M)

print("region area:  %.1f ha" % (region_geom.area(1).getInfo() / 1e4))
print("frame bbox:  ", [round(v, 4) for v in frame_bbox], f"(region bbox + {BUFFER_M:.0f} m)")

## 4 · One month = one view

`cadence` is monthly and **`end` is exclusive** (see `compositing.period_starts`), so
`2025-07-01 → 2025-08-01` is exactly one period and the run produces exactly one frame:
the **median** of every cloud-filtered Sentinel-2 scene over that month.

A few arguments earn their place for a single view:

- `preset="1080p"` upscales the native imagery to a real screen size — without it a
  100 m product like LST renders a few dozen pixels wide.
- `chart=False` turns off the inside-vs-outside time series; with one month there is no line to draw.
- `fps=1` — the MP4/GIF are a formality here, the PNG is the deliverable.
- `geotiffs=True` also exports the composited values, which is section 5.

In [ ]:
from gee_animation import animate

view = animate(
    REGION,
    sensor="sentinel2", index="ndvi",       # try landsat/lst, sentinel2/nbr, sentinel2/rgb
    start="2025-07-01", end="2025-08-01",   # end exclusive -> one month -> one frame
    buffer_m=BUFFER_M,
    preset="1080p",
    fps=1,
    chart=False,
    geotiffs=True,                          # the numbers behind the picture (section 5)
    out_dir="out/single_view",   # under out/, which the repo gitignores
    name="single_view_demo",
)
print(view.status)
view.frames

In [ ]:
from IPython.display import Image, display

still = sorted(p for p in view.frames if "_raw_" not in p)[0]
print(still)
display(Image(filename=still))

### What `render` added to the raw pixels

Reading that image means knowing what is *drawing* and what is *data*:

- **Colour** comes from the index's fixed `default_viz` (min, max, palette) in
  `products.INDICES` — **identical for every frame of every run of that index**, so two
  stills are comparable. Values outside the range are clipped, not rescaled.
- **Neutral grey** is no-data: cloud, shadow or missing scene. Earth Engine returns those
  pixels transparent and `render.apply_nodata` paints them grey on purpose — colorising
  them would read as bare soil (very low NDVI) and invent a clear-cut.
- **The thin outline** is your GeoJSON (`draw_region`); the header, colorbar, scale bar and
  credit line are drawn locally, after the fetch.
- The pixels were fetched at the product's native GSD and *then* upscaled by `preset`. The
  picture has more pixels than the measurement does.

## 5 · The same month as data

`geotiffs=True` wrote one GeoTIFF per observed frame to
`<out_dir>/geotiffs/<name>/<name>_<label>.tif`: the **composited float values** (NDVI here,
reflectance for `rgb`/`cir`) at native `scale`, in the resolved CRS. No palette, no clipping,
no furniture — this is the file you hand to QGIS, or reduce to a number for a paper.

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

tif = sorted(Path("out/single_view/geotiffs/single_view_demo").glob("*.tif"))[0]
with rasterio.open(tif) as src:
    data = src.read(1, masked=True)
    shape, res, crs = (src.width, src.height), src.res, src.crs

valid = 100.0 * data.count() / data.size
print(f"{tif.name}  |  {shape[0]}x{shape[1]} px @ {res[0]:.0f} m  |  {crs}")
print(f"NDVI  min {data.min():.3f}  mean {data.mean():.3f}  max {data.max():.3f}"
      f"  |  {valid:.1f}% valid pixels")

fig, ax = plt.subplots(figsize=(7, 6), dpi=110)
im = ax.imshow(data, vmin=0.0, vmax=1.0, cmap="YlGn")
fig.colorbar(im, ax=ax, shrink=0.8, label="NDVI")
ax.set_title(f"{tif.stem} — the values, unstyled")
ax.set_axis_off()
fig.tight_layout()

### Presenting is not measuring

The two sections rendered the same composite and you should expect them to *look* different:
the still is clipped to a palette and upscaled for a screen, the GeoTIFF is neither. Pull a
statistic from the still at your peril — take it from the GeoTIFF (or from
`view.metadata()`, which carries the per-frame AOI mean, scene count and cloud fraction
straight out of Earth Engine).

The repo's own split follows this line: `docs/rendering-products.md` for what each product
honestly resolves, and the `presenting-vs-measuring` skill for which knobs are free to change.

## 6 · Change one thing

| To get | Change |
|---|---|
| a different index | `index="nbr"` (burn/windthrow), `"ndwi"`, `"evi"`, `"rgb"` |
| land surface temperature | `sensor="landsat", index="lst"` — 100 m, so keep `preset` |
| a different month | `start`/`end` one month apart; `end` stays exclusive |
| a tighter or wider picture | `buffer_m=200` … `3000` |
| your own polygon | the `REGION` cell in section 3 |
| **an animation** | widen the dates: `start="2016-01-01", end="2026-01-01"`, `fps=2`, and drop `geotiffs` — that is a 120-frame run, and `cli.run()` with a YAML config in `config/` is the better tool for it |

An empty result (`RuntimeError: No sentinel2 ndvi imagery ...`) means the cloud filter
rejected every scene in that month: widen the dates or raise `region_max_cloud_percent`.